In [1]:
#Imports
import json
import numpy as np
import pandas as pd

In [2]:
#Read data from file
def assemble_data(filename, tweet_data):
    with open(filename, "r", encoding="utf-8") as file:
        content = file.read()

    prefix = 'window.YTD.tweets.part0 = '
    if content.startswith(prefix):
        json_data = content[len(prefix):].strip()
    else:
        json_data = content.strip()

    tweets_archive = json.loads(json_data)
    tweets = [item['tweet'] for item in tweets_archive]

    for t in tweets:
        tweet_data['Id'].append(str(t.get('id')))

        # Mark tweets without media for later pruning.
        if 'media' in t.get('entities'):
            tweet_data['Media'].append(t.get('entities')['media'][0]['media_url_https'])
        else:
            tweet_data['Media'].append(False)

        tweet_data['Likes'].append(int(t.get('favorite_count', 0)))
        tweet_data['Retweets'].append(int(t.get('retweet_count', 0)))

        tweet_data['Date'].append(t.get('created_at'))

        if t.get('full_text')[:2] != 'RT':  # Check if tweet is a Repost
            words = t.get('full_text').split()
            if words and words[-1].startswith('https://t.co/'):
                words = words[:-1]
            tweet_data['Text'].append(' '.join(words))
            tweet_data['IsRepost'].append('Not a Repost')
        else:
            tweet_data['Text'].append(t.get('full_text'))
            tweet_data['IsRepost'].append('Is a Repost')

        # Quote-tweet detection
        if len(t.get('full_text').split()) > 1:
            if 'https://' in t.get('full_text').split()[-2]:
                tweet_data['IsQuote'].append('Is a Quote')
            else:
                tweet_data['IsQuote'].append('Not a Quote')
        else:
            tweet_data['IsQuote'].append('Not a Quote')

        hashtags = t.get('entities')['hashtags']
        if hashtags:
            tweet_data['Hashtags'].append([h['text'] for h in hashtags])
        else:
            tweet_data['Hashtags'].append(None)

        if 'in_reply_to_user_id' in t:
            tweet_data['Reply'].append('Is a Reply')
        else:
            tweet_data['Reply'].append('Not a Reply')


In [3]:
tweet_data = {
        "Id": [],
        "Media": [],
        "Likes": [],
        "Retweets": [],
        "Date": [],
        "Text": [],
        "Hashtags": [],
        "Reply": [],
        "IsRepost": [],
        "IsQuote": [],
    }
#Process Both Data Sets
assemble_data("tweets1.js", tweet_data)
assemble_data("tweets2.js", tweet_data)

#Create Table
df = pd.DataFrame(tweet_data)

print(f'Shape: {df.shape}')


Shape: (27745, 10)


In [4]:
df.head()

,Id,Media,Likes,Retweets,Date,Text,Hashtags,Reply,IsRepost,IsQuote
0,2095360617318080538,https://pbs.twimg.com/media/HRQVxekaoAAfdCt.jpg,0,0,Thu Sep 03 03:57:55 +0000 2026,RT @rokurin4297: Local priest gets an arrow to...,None,Not a reply,Is a Repost,Not a Quote
1,2090539507632246846,https://pbs.twimg.com/amplify_video_thumb/2060...,0,0,Thu Aug 20 20:40:33 +0000 2026,RT @Podnitrix: Waking up to take a piss at 3 a...,None,Not a reply,Is a Repost,Not a Quote
2,2090223661533762045,https://pbs.twimg.com/media/HQD67gcX0AAh6U2.jpg,0,0,Wed Aug 19 23:45:30 +0000 2026,RT @katekyo98154: Ego https://t.co/jpBiQL585a,None,Not a reply,Is a Repost,Not a Quote
3,2090076316716736674,https://pbs.twimg.com/media/HQD67gcX0AAh6U2.jpg,4,4,Wed Aug 19 14:00:00 +0000 2026,Ego,None,Not a reply,Not a Repost,Not a Quote
4,2089896522292834366,https://pbs.twimg.com/media/HQDNwq3WYAE7j6U.jpg,5,1,Wed Aug 19 02:05:33 +0000 2026,Perugius Sketch from memory,None,Not a reply,Not a Repost,Not a Quote


In [5]:
#Prune Data

df = df[df['IsRepost'] != "Is a Repost" ]
df = df[df['IsQuote'] != "Is a Quote" ]
df = df[df['Reply'] == "Not a reply" ]
df = df[df['Media'] != False ]

#Colomuns are no longer needed for later analysis.
df = df.drop(columns="IsRepost")
df = df.drop(columns="IsQuote")
df = df.drop(columns="Reply") 

print(f'Shape: {df.shape}')

Shape: (989, 7)


In [6]:
df.head()

,Id,Media,Likes,Retweets,Date,Text,Hashtags
3,2090076316716736674,https://pbs.twimg.com/media/HQD67gcX0AAh6U2.jpg,4,4,Wed Aug 19 14:00:00 +0000 2026,Ego,None
4,2089896522292834366,https://pbs.twimg.com/media/HQDNwq3WYAE7j6U.jpg,5,1,Wed Aug 19 02:05:33 +0000 2026,Perugius Sketch from memory,None
6,2085391698402955589,https://pbs.twimg.com/media/HPAaZFvWUAAsjBw.jpg,1,1,Thu Aug 06 15:45:00 +0000 2026,Sketch,None
8,2085192279082320247,https://pbs.twimg.com/media/HPAXN2FXYAAiFaK.jpg,5,4,Thu Aug 06 02:32:34 +0000 2026,Recent Character design I did for fun,None
12,2083635519108403559,https://pbs.twimg.com/media/HOqPb_pXAAAX5GR.jpg,1,3,Sat Aug 01 19:26:34 +0000 2026,Arthur,None


In [7]:
#Export to CSV
df.to_csv('Tweets_Data.csv', index=False)

#Because some images are not drawings, I need to remove those from the set aswell. However, there is no feasible way to distinguish the two,
#So it must be done manually


Unfornutately, I realized after sifting through the data that I didn't scan for Quote Tweets. So I went back to the assemble_data function, and rewrote it to check for that. After pruning the inital data set which possessed the quote tweets, I need to check it against my data set I removed all the non-Art related posts from. Then, by creating a data frame where the Id of the posts match in both, I can have the data that I need. 

In [8]:
#check cleaned data
df2 = pd.read_csv('Tweets_Data2.csv',dtype={'Id': 'str'}) #Prevent it turning Tweet Id into a Floating Point Number
df2 = df2.dropna(how='all')
df2.reset_index(drop=True, inplace=True)
df.reset_index(drop=True, inplace=True)


In [9]:
# Fix #1: .copy() prevents SettingWithCopyWarning on later assignments
df3 = df[df['Id'].isin(df2['Id'])].copy()  # cross-reference with manually curated set
print(df3.shape)


(573, 7)


In [10]:
#Reformat Date
df3['Date'] = pd.to_datetime(df3['Date'], format= "%a %b %d %H:%M:%S %z %Y")
df3['Date']

0     2026-08-19 14:00:00+00:00
1     2026-08-19 02:05:33+00:00
2     2026-08-06 15:45:00+00:00
3     2026-08-06 02:32:34+00:00
4     2026-08-01 19:26:34+00:00
                 ...           
983   2021-07-18 19:56:04+00:00
984   2021-07-18 18:49:29+00:00
986   2021-07-15 18:55:35+00:00
987   2021-07-15 04:26:04+00:00
988   2021-07-09 02:58:58+00:00
Name: Date, Length: 573, dtype: datetime64[ns, UTC]

In [11]:
df3.head()


,Id,Media,Likes,Retweets,Date,Text,Hashtags
0,2090076316716736674,https://pbs.twimg.com/media/HQD67gcX0AAh6U2.jpg,4,4,2026-08-19 14:00:00+00:00,Ego,None
1,2089896522292834366,https://pbs.twimg.com/media/HQDNwq3WYAE7j6U.jpg,5,1,2026-08-19 02:05:33+00:00,Perugius Sketch from memory,None
2,2085391698402955589,https://pbs.twimg.com/media/HPAaZFvWUAAsjBw.jpg,1,1,2026-08-06 15:45:00+00:00,Sketch,None
3,2085192279082320247,https://pbs.twimg.com/media/HPAXN2FXYAAiFaK.jpg,5,4,2026-08-06 02:32:34+00:00,Recent Character design I did for fun,None
4,2083635519108403559,https://pbs.twimg.com/media/HOqPb_pXAAAX5GR.jpg,1,3,2026-08-01 19:26:34+00:00,Arthur,None


In [12]:
#Data for Analysis
df3.to_csv('Cleaned_Tweets_Data.csv', index=False)
